# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library. The dataset contains clinical and molecular data for cancer survivors with second primary colorectal cancer, including MSI-H status and anatomical characteristics.

### Dataset Source
The dataset source is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the `mlcroissant` library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (use .metadata directly, do not subscript)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Identifier: {dataset.metadata.identifier}\n")
print(f"Date published: {dataset.metadata.datePublished}\n")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, let's list the available record sets and their fields using their `@id`s.

In [ ]:
# The mlcroissant API exposes record sets via .record_sets, each with a unique `@id`.
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields (with @id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    print("")

Let's preview a sample record for each record set by `@id`.

In [ ]:
for rs in record_sets:
    print(f"Sample record from RecordSet '{rs.name}' (@id: {rs.id}):")
    records = list(dataset.records(record_set=rs.id))
    if records:
        print(records[0])
    else:
        print("No records found.")
    print("\n---\n")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. For reference, all entities are referenced by their Croissant `@id` fields.

Below, we demonstrate extracting all record sets.

In [ ]:
# Build up a DataFrame for each record set, indexed by its @id
dataframes = {}

record_set_ids = [rs.id for rs in dataset.record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id} | Shape: {df.shape}")
    else:
        print(f"RecordSet @id: {record_set_id} has no records.")

Let's examine the columns (fields) and a preview of the main data table. We'll choose the principal record set (typically the clinical cases table). If there are multiple record sets, select the one with the largest number of columns or records.

In [ ]:
# Pick the record set with the most fields as the main analysis table
main_rs_id = None
max_columns = 0
for rs_id, df in dataframes.items():
    if len(df.columns) > max_columns:
        main_rs_id = rs_id
        max_columns = len(df.columns)

print(f"Chosen main RecordSet @id: {main_rs_id}")
print("Column names (by field @id):")
print(list(dataframes[main_rs_id].columns))
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing and exploration using Croissant field `@id` references. We'll select a numeric field and a categorical (group) field by `@id` for demonstration below.

*Replace `<numeric_field_id>` and `<group_field_id>` with field IDs identified above, as appropriate.*

In [ ]:
# --- Specify which field @id to use for numeric analysis and grouping ---
# Use the output above to identify a relevant numeric field (e.g., patient age, diagnosis interval)
# For demonstration, we attempt to auto-select plausible IDs; you may override with field IDs from cell above.
num_field = None
group_field = None
for col in dataframes[main_rs_id].columns:
    if (
        'age' in col.lower()
        or 'interval' in col.lower()
        or 'years' in col.lower()
    ) and num_field is None:
        num_field = col
    if (
        'sex' in col.lower() or 'gender' in col.lower() or 'anatomical' in col.lower()
    ) and group_field is None:
        group_field = col

print(f"Numeric field @id: {num_field}")
print(f"Group field @id: {group_field}")

# Ensure required fields exist
df = dataframes[main_rs_id].copy()
if num_field is not None and pd.api.types.is_numeric_dtype(df[num_field]):
    threshold = df[num_field].mean()  # Use mean as example threshold
    filtered_df = df[df[num_field] > threshold]
    print(f"Filtered records with {num_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{num_field}_normalized"] = (
        (filtered_df[num_field] - filtered_df[num_field].mean()) / filtered_df[num_field].std()
    )
    print(f"\nNormalized {num_field} for filtered records:")
    print(filtered_df[[num_field, f"{num_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[num_field].mean().reset_index()
        print(f"\nGrouped mean {num_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field suitable for analysis found in the selected record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset, using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Numeric field histogram
if num_field is not None and pd.api.types.is_numeric_dtype(df[num_field]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[num_field], bins=15, kde=True)
    plt.title(f"Distribution of {num_field}")
    plt.xlabel(num_field)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group (if fields exist)
if (
    group_field is not None
    and num_field is not None
    and group_field in df.columns
    and pd.api.types.is_numeric_dtype(df[num_field])
):
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=num_field, data=df)
    plt.title(f"{num_field} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load FAIR²-compliant clinical datasets using the Croissant schema.
- Explore available record sets, fields, and reference all entities by their `@id`s.
- Extract and analyze main clinical records as a DataFrame using only Croissant identifiers.
- Perform EDA including filtering, normalization, grouping, and basic visualizations with references to Croissant IDs.

This workflow ensures your data exploration is robust, reproducible, and fully traceable to the schema, facilitating interoperability and FAIR-aligned data science.